# 🎙️ EIT Audio Transcription Pipeline
### Audio-to-Text for Second/Additional Language Learner Data

**Project Goal:** Convert Spanish EIT learner audio responses into accurate text transcriptions,  
achieving ≥90% agreement with experienced human transcribers.

---

## Pipeline Overview

```
Raw Audio (.wav/.mp3)
       ↓
  [1] Preprocessing     → Noise reduction, normalization, silence trimming
       ↓
  [2] Transcription     → Whisper ASR (fine-tuned for learner speech)
       ↓
  [3] Post-processing   → Learner error correction, disfluency removal
       ↓
  Output: CSV / JSON with transcriptions + confidence scores
```


## 📦 Setup & Imports

In [ ]:
# Install required packages (run once)
# !pip install openai-whisper numpy scipy matplotlib pandas

import sys
sys.path.insert(0, 'src')

import numpy as np
import wave
import struct
import json
import os
import re
from pathlib import Path

from transcription_pipeline import (
    AudioPreprocessor,
    WhisperTranscriber,
    LearnerLanguagePostProcessor,
    EITPipeline,
    TranscriptionResult
)

print('✅ All modules loaded successfully')

## 🎵 Step 1: Generate / Load Dummy Audio

Since we don't have real learner data, we generate synthetic `.wav` files  
that simulate non-native Spanish speech patterns.

In [ ]:
AUDIO_DIR = 'audio_samples'
os.makedirs(AUDIO_DIR, exist_ok=True)

SAMPLE_RATE = 16000
EIT_SENTENCES = [
    'El niño come una manzana roja',
    'La profesora habla con los estudiantes',
    'Nosotros vivimos en una ciudad grande',
    'Ella siempre estudia por la noche',
    'Los pájaros cantan en el jardín',
]

def generate_dummy_audio(filepath, duration=2.5, noise_level=0.07):
    """Generate a synthetic WAV file simulating learner speech."""
    t = np.linspace(0, duration, int(SAMPLE_RATE * duration))
    signal  = 0.3 * np.sin(2 * np.pi * 180 * t)
    signal += 0.2 * np.sin(2 * np.pi * 350 * t)
    signal += 0.1 * np.sin(2 * np.pi * 700 * t)
    mod = 0.5 + 0.5 * np.sin(2 * np.pi * 3.5 * t)  # syllable rhythm
    signal = signal * mod + noise_level * np.random.randn(len(t))
    signal = signal / np.max(np.abs(signal)) * 0.8
    
    with wave.open(filepath, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        frames = struct.pack('<' + 'h' * len(signal), *[int(s * 32767) for s in signal])
        wf.writeframes(frames)

# Generate 3 learners × 5 sentences = 15 files
learner_ids = ['learner_01', 'learner_02', 'learner_03']
count = 0
for lid in learner_ids:
    for i in range(len(EIT_SENTENCES)):
        fname = f'{AUDIO_DIR}/{lid}_sentence_{i+1:02d}.wav'
        noise = np.random.uniform(0.03, 0.12)
        dur   = np.random.uniform(1.8, 3.5)
        generate_dummy_audio(fname, duration=dur, noise_level=noise)
        count += 1

files = sorted(Path(AUDIO_DIR).glob('*.wav'))
print(f'✅ Generated {count} audio files:')
for f in files[:5]:
    print(f'   {f.name}')
print(f'   ... and {len(files)-5} more')

## 🔧 Step 2: Audio Preprocessing

Demonstrates noise gating, normalization, and silence trimming on one file.

In [ ]:
preprocessor = AudioPreprocessor(sample_rate=16000, silence_threshold=0.01)

# Test on first file
test_file = str(files[0])
raw_samples, sr = preprocessor.load_audio(test_file)
print(f'Raw audio: {len(raw_samples)} samples, {len(raw_samples)/sr:.2f}s, SR={sr}Hz')

# Process step by step
normalized = preprocessor.normalize(raw_samples)
gated      = preprocessor.noise_gate(normalized)
trimmed    = preprocessor.trim_silence(gated)

print(f'After normalization: peak={np.max(np.abs(normalized)):.3f}')
print(f'After noise gate:    non-zero frames={np.count_nonzero(gated)}')
print(f'After trim silence:  {len(trimmed)/sr:.2f}s (was {len(raw_samples)/sr:.2f}s)')
print('✅ Preprocessing complete')

## 🤖 Step 3: Transcription with Whisper

**Demo mode** is active by default (no install needed).  
To use real Whisper: `pip install openai-whisper` then set `use_demo=False`.

In [ ]:
# Initialize transcriber
transcriber = WhisperTranscriber(
    model_size='medium',
    language='es',
    use_demo=True    # Change to False + pip install openai-whisper for real transcription
)

# Transcribe first 5 files
print('File'.ljust(40), 'Transcription'.ljust(50), 'Conf')
print('-' * 95)

sample_results = []
for f in files[:5]:
    match = re.search(r'sentence_(\d+)', f.stem)
    sidx  = int(match.group(1)) - 1 if match else None
    text, conf = transcriber.transcribe(str(f), sentence_index=sidx)
    print(f.name[:38].ljust(40), text[:48].ljust(50), f'{conf:.2f}')
    sample_results.append({'file': f.name, 'text': text, 'conf': conf})

## 🛠️ Step 4: Post-Processing — Learner Error Correction

In [ ]:
postprocessor = LearnerLanguagePostProcessor(
    apply_accent_restoration=True,
    remove_disfluencies=True
)

# Test with learner error examples
test_cases = [
    'eh el niño bamos a la escuela',         # disfluency + b/v confusion
    'la profesora habla and los estudiantes', # L1 transfer (English 'and')
    'nosotros vivimos in ciudad grande',      # L1 transfer (English 'in')
    'ella siempre estudia por la noche',      # correct
]

print('Input'.ljust(50), 'Corrected'.ljust(50), 'Flags')
print('-' * 130)
for t in test_cases:
    corrected, flags = postprocessor.correct(t)
    print(t[:48].ljust(50), corrected[:48].ljust(50), str(flags)[:30])

## 🚀 Step 5: Run Full Pipeline on All Audio Files

In [ ]:
# Run full pipeline
pipeline = EITPipeline(
    output_dir='outputs',
    use_demo=True,           # Set False for real Whisper
    whisper_model='medium'
)

results = pipeline.process_directory(AUDIO_DIR)
print(f'\n✅ Processed {len(results)} files')

## 📊 Step 6: Evaluate Results (90% Agreement Target)

In [ ]:
# Save output files
saved = pipeline.save_results(format='both')
print('Saved files:')
for fmt, path in saved.items():
    print(f'  {fmt}: {path}')

# Generate and display report
report = pipeline.generate_report()
print('\n' + '='*55)
print('  PIPELINE EVALUATION REPORT')
print('='*55)
for k, v in report.items():
    status = ''
    if k == 'estimated_word_agreement' and isinstance(v, float):
        status = ' ✅' if v >= 0.9 else ' ❌ (needs improvement)'
    if k == 'meets_90pct_target':
        status = ' ✅' if v else ' ❌'
    print(f'  {k:<38} {v}{status}')
print('='*55)

## 📋 Step 7: View Results Table

In [ ]:
try:
    import pandas as pd
    df = pd.read_csv('outputs/transcriptions.csv')
    display(df[['learner_id','sentence_id','corrected_transcription',
                'confidence_score','duration_seconds','flags']].head(10))
except ImportError:
    # Fallback without pandas
    import csv
    with open('outputs/transcriptions.csv') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            if i >= 10: break
            print(f"{row['learner_id']:12} | {row['sentence_id']:30} | "
                  f"{row['corrected_transcription']:45} | conf={row['confidence_score']}")

## 🔬 Step 8: Fine-Tuning Guide (For Real Data)

When you have real learner audio data, follow these steps to fine-tune Whisper:

```python
# 1. Prepare dataset
#    CSV with: audio_path | reference_transcription | learner_id | L1_background

# 2. Fine-tune using HuggingFace Transformers
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer

model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-medium')
processor = WhisperProcessor.from_pretrained('openai/whisper-medium', language='Spanish')

# 3. Training arguments (recommended settings for learner speech)
training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-eit-finetuned',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    generation_max_length=225,
    predict_with_generate=True,
    fp16=True,   # Use False on CPU
)

# 4. Train and save
trainer = Seq2SeqTrainer(model=model, args=training_args, ...)
trainer.train()
model.save_pretrained('./whisper-eit-finetuned')

# 5. Use fine-tuned model in pipeline
pipeline = EITPipeline(
    use_demo=False,
    model_path='./whisper-eit-finetuned'
)
```
